In [1]:
#!pip install -U langchain langchain-core langchain-openai langgraph tavily
from typing import Literal, Optional, TypedDict, List, Literal
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command, RetryPolicy
from google.colab import drive
from tavily import TavilyClient
from concurrent.futures import ThreadPoolExecutor
import pandas as pd

drive.mount('/content/drive')
file_path1 = "/content/drive/MyDrive/Final_proj_GA/capitals.xlsx"
file_path2 = "/content/drive/MyDrive/Final_proj_GA/mountains.xlsx"
df1 = pd.read_excel(file_path1)
df2 = pd.read_excel(file_path2)

from google.colab import userdata
api_key = userdata.get('OPENROUTER_APIKEY')
tavily_api_key = userdata.get('TAVILY_APIKEY')

tavily = TavilyClient(api_key=tavily_api_key)
llm = ChatOpenAI(model="openai/gpt-6-luna", temperature=0.1, api_key=api_key, base_url="https://openrouter.ai/api/v1")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
df1.head()

,Capital_From,Country_From,Capital_To,Country_To,distance
0,Kyiv,Ukraine,Paris,France,NaN
1,London,United Kingdom,Rome,Italy,NaN
2,Paris,France,Berlin,Germany,NaN
3,Berlin,Germany,Vienna,Austria,524.0
4,Warsaw,Poland,Kyiv,Ukraine,NaN


In [3]:
df2.head()

,Mountain,Country,height,distance
0,Mount Everest,Nepal/China,NaN,0
1,K2,Pakistan/China,NaN,0
2,Kangchenjunga,Nepal/India,NaN,0
3,Lhotse,Nepal/China,NaN,0
4,Makalu,Nepal/China,NaN,0


In [4]:
class Agent_state(TypedDict):
  table:pd.DataFrame
  question: str
  search_questions: list
  search_answers: list
  missing_column: str
  file_path: str

class Answer(TypedDict):
  answer: float


In [10]:
def analyze(state:Agent_state) -> Command[Literal['search']]:
  """
  аналізує питання та переформульовує та конкретизує його на основі даних зі сховища.
  """
  tasks = []
  for task in range(len(state['table'])):
    rows = []
    for i, column in enumerate(state['table'].columns):
      string = f"Колонка {i}, назва: {column}, вміст: {state['table'][column][task]}"
      rows.append(string)
    f_string = '\\n'.join([f"- {thing}" for thing in rows])
    tasks.append(f_string)

  def process_one_a(task):
    prompt = f"""
    Ти аналізатор, твоя задача проаналізувати питання та наявні колонки, та ще одну пусту колонку,
    після чого сформувати пошукове питання для пошуку в мережі, приклад:

    Питання: знайди пряму відстань між столицями в км для колонки distance.
    Колонки:
    - Колонка 1, назва: Capital_From, вміст: Kyiv
    - Колонка 2, назва: Country_From, вміст: Ukraine
    - Колонка 3, назва: Capital_To, вміст: Paris
    - Колонка 4, назва: Country_To, вміст: France
    - Колонка 5, назва: distance, вміст: None
    Твоя відповідь: Яка відстань між Києвом та Парижем в кілометрах?

    А тепер реальна задача:

    Питання: {state['question']}
    Колонки: {task}
    Твоя відповідь:
    """
    answer = llm.invoke(prompt)
    return answer.content

  def parallel_processing(max_w, tasks):
    with ThreadPoolExecutor(max_workers=max_w) as executor:
      result = list(executor.map(process_one_a, tasks))
    return result

  rows = parallel_processing(max_w=5, tasks=tasks)
  return Command(update={'search_questions':rows}, goto="search")

def search(state: Agent_state) -> Command[Literal['update_table']]:
    """
    Шукає відповіді на питання, сформульовані функцією analyze.
    """

    def process_one(question):
      try:
          response = tavily.search(
              query=question,
              search_depth="advanced",
              max_results=5,
              topic="general",
              include_answer="advanced"
          )

          results = response.get("results", [])

          results = [
              r for r in results
              if r.get("score", 0) >= 0.3
          ]

          results = sorted(
              results,
              key=lambda x: x.get("score", 0),
              reverse=True
          )[:5]

          context = "\n\n".join(
              f"""
              Title: {r.get('title', '')}
              URL: {r.get('url', '')}
              Content: {r.get('content', '')}
              """
              for r in results
          )

          return context

      except Exception as e:
          print(f"Search error: {e}")
          return ""

    def parallel_processing(questions, max_workers=5):

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            results = list(executor.map(process_one, questions))

        return results

    rows = parallel_processing(
        questions=state['search_questions'],
        max_workers=5
    )

    return Command(
        update={'search_answers': rows},
        goto='update_table'
    )

def update_table(state: Agent_state) -> Command[Literal[END]]:
    """
    Формалізує відповіді, вносить правки до таблиці та зберігає її.
    """

    llm_structured = llm.with_structured_output(Answer)

    def process_one_ut(t):

        prompt = f"""
        Ти аналізатор. Твоя задача відповісти на питання
        використовуючи наведений контекст.

        Якщо контекст не містить відповіді або відповідь невідома,
        поверни 'not_found'. Не вигадуй відповіді та не використовуй 0.

        Питання: {state['search_questions'][t]}

        Контекст: {state['search_answers'][t]}

        Твоя відповідь:
        """

        response = llm_structured.invoke(prompt)

        return response


    def parallel_processing(ts, max_w=5):

        with ThreadPoolExecutor(max_workers=max_w) as executor:
            results = list(executor.map(process_one_ut, ts))

        return results

    ts = list(range(len(state['search_answers'])))

    rows = parallel_processing(ts=ts, max_w=5)


    try:
        for t in range(len(rows)):

            print(rows[t])

            answer = rows[t]['answer']

            if answer == 'not_found' or answer <= 0:
                state['table'].loc[t, state['missing_column']] = float('nan')
            else:
                state['table'].loc[t, state['missing_column']] = answer

        state['table'].to_excel(state['file_path'], index=False)

    except Exception as e:
        print(f'update_table faced error: {e}')

    return Command(update={}, goto=END)


In [6]:
def build_agent():
  # Агент
  workflow = StateGraph(Agent_state)

  # nodes
  workflow.add_node('analyze', analyze)
  workflow.add_node('search', search)
  workflow.add_node('update_table', update_table)

  # edges

  workflow.add_edge(START, 'analyze')
  workflow.add_edge('analyze', 'search')
  workflow.add_edge('search', 'update_table')
  workflow.add_edge('update_table', END)

  #memory = MemorySaver()
  app = workflow.compile()
  return app

In [7]:
def process_excel(file_path:str, task_desc:str, missing_column:str, experiment_id:str):
  # Функція взаємодії з агентом
  agent = build_agent()
  initial_state = {
      'file_path': file_path,
      'question': task_desc,
      'missing_column': missing_column,
      'table': pd.read_excel(file_path),
      'search_questions': [],
      'search_answers': []
      }
  config = {
    "configurable": {
        "thread_id": experiment_id
    }
  }
  agent.invoke(initial_state, config)

In [8]:
process_excel(
    file_path="/content/drive/MyDrive/Final_proj_GA/capitals.xlsx",
    task_desc="знайди пряму відстань між столицями в км для колонки distance",
    missing_column="distance",
    experiment_id='1'
)

{'answer': 2031}
{'answer': 1431}
{'answer': 878.08}
{'answer': 523.57}
{'answer': 688.47}
{'answer': 1363}
{'answer': 502}
{'answer': 215}
{'answer': 416}
{'answer': 566}


In [9]:
process_excel(
    file_path="/content/drive/MyDrive/Final_proj_GA/mountains.xlsx",
    task_desc="додай висоту гір у метрах до колонки height",
    missing_column="height",
    experiment_id='2'
)

{'answer': 8848.86}
{'answer': 8611}
{'answer': 8586}
{'answer': 8516}
{'answer': 8485}
{'answer': 8188}
{'answer': 8167}
{'answer': 8163}
{'answer': 8126}
{'answer': 8091}
